In [ ]:
import torch
from torch import nn

# ── 1. MAKE SOME DATA ───────────────────────────────────────────────────────
# Each row is a sequence of 4 past flips.
# The label is what came next (1=Heads, 0=Tails).
#
# Example:  [1, 0, 1, 1]  →  next flip was 0 (Tails)

sequences = torch.tensor([
    [1, 0, 1, 1],
    [0, 0, 0, 0],
    [1, 1, 1, 1],
    [0, 1, 0, 1],
    [1, 0, 0, 0],
    [0, 0, 1, 1],
    [1, 1, 0, 0],
    [0, 1, 1, 0],
], dtype=torch.float32)

labels = torch.tensor([0, 1, 0, 1, 1, 1, 0, 1], dtype=torch.float32)


# ── 2. BUILD THE MODEL ───────────────────────────────────────────────────────
# Same pattern as the PyTorch tutorial (nn.Module + nn.Sequential).
# Input:  4 past flips
# Hidden: 16 neurons  (small — our dataset is tiny)
# Output: 1 number → squeeze through Sigmoid → probability of Heads

class AutoregressiveCoinFlip(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(4, 16),   # 4 past flips → 16 hidden neurons
            nn.ReLU(),           # non-linearity (same as tutorial)
            nn.Linear(16, 1),   # 16 hidden → 1 output
            nn.Sigmoid()         # squash to [0, 1] = probability of Heads
        )

    def forward(self, x):
        return self.network(x).squeeze()   # remove extra dimension


model = AutoregressiveCoinFlip()
print("Model structure:")
print(model)


# ── 3. TRAIN THE MODEL ───────────────────────────────────────────────────────
loss_fn   = nn.BCELoss()                        # Binary Cross-Entropy for 0/1 output
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

print("\nTraining...")
for epoch in range(500):
    # Forward pass: predict probabilities
    predictions = model(sequences)

    # Compute how wrong we are
    loss = loss_fn(predictions, labels)

    # Backward pass: adjust weights
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 100 == 0:
        print(f"  Epoch {epoch+1}/500 | Loss: {loss.item():.4f}")


# ── 4. USE THE MODEL ─────────────────────────────────────────────────────────
print("\n── Predictions ──")
model.eval()   # turn off training mode
with torch.no_grad():
    for seq, true_label in zip(sequences, labels):
        prob_heads = model(seq.unsqueeze(0)).item()
        predicted  = "Heads" if prob_heads > 0.5 else "Tails"
        actual     = "Heads" if true_label == 1   else "Tails"
        flips      = ["H" if f == 1 else "T" for f in seq.int().tolist()]
        print(f"  History: {flips}  →  P(Heads)={prob_heads:.2f}  "
              f"Predicted: {predicted}  Actual: {actual}")

# ── 5. PREDICT ON A NEW SEQUENCE ─────────────────────────────────────────────
print("\n── New Sequence ──")
new_seq = torch.tensor([[1, 1, 0, 0]], dtype=torch.float32)
with torch.no_grad():
    prob = model(new_seq).item()
print(f"  History: [H, H, T, T]  →  P(next flip = Heads) = {prob:.2f}")

Model structure:
AutoregressiveCoinFlip(
  (network): Sequential(
    (0): Linear(in_features=4, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=1, bias=True)
    (3): Sigmoid()
  )
)

Training...
  Epoch 100/500 | Loss: 0.0383
  Epoch 200/500 | Loss: 0.0063
  Epoch 300/500 | Loss: 0.0026
  Epoch 400/500 | Loss: 0.0014
  Epoch 500/500 | Loss: 0.0009

── Predictions ──
  History: ['H', 'T', 'H', 'H']  →  P(Heads)=0.00  Predicted: Tails  Actual: Tails
  History: ['T', 'T', 'T', 'T']  →  P(Heads)=1.00  Predicted: Heads  Actual: Heads
  History: ['H', 'H', 'H', 'H']  →  P(Heads)=0.00  Predicted: Tails  Actual: Tails
  History: ['T', 'H', 'T', 'H']  →  P(Heads)=1.00  Predicted: Heads  Actual: Heads
  History: ['H', 'T', 'T', 'T']  →  P(Heads)=1.00  Predicted: Heads  Actual: Heads
  History: ['T', 'T', 'H', 'H']  →  P(Heads)=1.00  Predicted: Heads  Actual: Heads
  History: ['H', 'H', 'T', 'T']  →  P(Heads)=0.00  Predicted: Tails  Actual: Tails
  Histo